In [2]:
import sys
sys.path.append("../")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit.quantum_info import Statevector, SparsePauliOp
from qitp.engine.unitary_approximation import UnitaryApproximation
from qitp.engine.propagation import QITEPropagation
from qitp.engine.cqite_run import run_qite
from qitp.engine.prepare_state import constraint_rotation, prepare_state, density_from_rotations, support_only
from qitp.simulation.backend import SimulatorBackend

In [4]:
from qitp.ansatz.pauli_basis import (
    PauliBasis
)
from qitp.ansatz.commutator_basis import (
    CommutatorBasis
)

In [5]:
depth=2
dt=1e-1/2
nqubits = 6

In [6]:
nsteps=45
target=0.4

energies=[]
fidelities=[]
errors=[]
magnetizations=[]

In [7]:
from qiskit.quantum_info import SparsePauliOp

Jzz = 0.7
Jxx = 0.4
n = nqubits

edges = [
    (1,2),
    (1,3),
    (2,3),
    (2,4),
    (3,4),
    (0,2),
    (0,4),
    (0,5),
    (4,5),
]

paulis = []
coeffs = []

for u, v in edges:
    s = ["I"] * n
    s[u] = "Z"
    s[v] = "Z"
    paulis.append("".join(s))
    coeffs.append(Jzz)

    s = ["I"] * n
    s[u] = "X"
    s[v] = "X"
    paulis.append("".join(s))
    coeffs.append(Jxx)

H = SparsePauliOp(paulis, coeffs=coeffs)

In [8]:
from qiskit.quantum_info import SparsePauliOp

n = nqubits

constraints = []

for i in range(n):
    s = ["I"] * n
    s[i] = "Y"
    constraints.append(
        SparsePauliOp(["".join(s)], coeffs=[1.0])
    )

Identity = SparsePauliOp(["I" * n], coeffs=[1.0])

In [9]:
Mz = constraints

In [10]:
H_eff = H + sum(Mz[1:], start=Mz[0])# + Identity

In [11]:
rotations = []

for constraint in Mz:

    rotations.append(
        constraint_rotation(
            constraint,
            target
        )
    )

In [12]:
rho = density_from_rotations(nqubits, rotations, degree=2)

seeds = support_only(rho)

In [13]:
basis = CommutatorBasis.from_seed(hamiltonian=H_eff, seeds=seeds, depth=depth)

In [14]:
len(basis.paulis)

217

In [15]:
circuit = prepare_state(
    nqubits,
    rotations
)

In [16]:
circuit = circuit.decompose()

In [17]:
approx = UnitaryApproximation(basis, trot=3)

backend = SimulatorBackend()

prop = QITEPropagation(
    H,
    backend,
    approx
)

prop.constraints = [
    Identity]
prop.constraints.extend(Mz)

In [18]:
from qiskit.quantum_info import Statevector

psi = Statevector.from_instruction(circuit)

In [ ]:
results = run_qite(
    prop,
    circuit,
    H,
    Mz,
    dt,
    nsteps,
    state=psi #Remove for exponential advantage
)

In [ ]:
energies = results["energies"]
magnetizations = results["magnetizations"] #[np.sum(m) for m in results["magnetizations"]]
circuit = results["circuit"]

In [ ]:
print(np.round(energies[-1],3))

In [ ]:
print(np.round(magnetizations[-1],3))

In [ ]:
plt.plot(energies); plt.title('Energy'); plt.grid(); plt.show()

In [ ]:
plt.plot(magnetizations); plt.title("Magnetization"); plt.grid(); plt.show()

In [62]:
evals = np.linalg.eigvalsh(
    H.to_matrix()
)

print("Exact spectrum")

print(np.round(evals,2))

Exact spectrum
[-4.18 -3.87 -3.16 -2.91 -2.84 -2.71 -2.68 -2.66 -2.54 -2.54 -2.43 -2.41
 -2.34 -2.28 -2.23 -2.   -1.95 -1.89 -1.71 -1.54 -1.34 -1.28 -1.1  -1.08
 -0.89 -0.83 -0.78 -0.75 -0.72 -0.72 -0.64 -0.6  -0.59 -0.49 -0.32 -0.17
 -0.16 -0.05  0.33  0.63  0.79  0.92  0.96  1.01  1.07  1.17  1.46  1.77
  1.78  1.81  1.81  1.88  2.06  2.54  2.59  2.75  3.08  3.25  3.77  3.78
  4.17  4.78  6.54  6.67]


In [64]:
def compute_betas_dense(psi, Hm, constraints):

    m = len(constraints)

    gamma = np.zeros(m)
    Gmat = np.zeros((m,m))

    Hpsi = Hm @ psi

    for i, Bi in enumerate(constraints):

        Bipsi = Bi @ psi

        gamma[i] = -np.real(
            np.vdot(Hpsi, Bipsi)
        )

        for j, Bj in enumerate(constraints):

            Gmat[i,j] = np.real(
                np.vdot(
                    Bipsi,
                    Bj @ psi
                )
            )

    return np.linalg.pinv(Gmat) @ gamma

In [66]:
Hm = H.to_matrix()

In [68]:
C = []

for i in range(nqubits):

    p = ["I"] * nqubits
    p[i] = "Y"

    C.append(
        SparsePauliOp(
            ["".join(p)],
            coeffs=[1.0]
        ).to_matrix()
    )

I = np.eye(Hm.shape[0])

constraints_mat = [I]
constraints_mat.extend(C)

In [70]:
from scipy.linalg import expm

def cite_step(
    psi,
    Hm,
    constraints_mat,
    dt,
    nsub=100
):

    h = dt / nsub

    for _ in range(nsub):

        beta = compute_betas_dense(
            psi,
            Hm,
            constraints_mat
        )

        G = Hm.copy()

        for b, B in zip(beta, constraints_mat):
            G += b * B

        psi = expm(-h * G) @ psi
        psi /= np.linalg.norm(psi)

    return psi

In [ ]:
energies_exact = []
constraints_history = [
    [] for _ in C
]

psi_exact = psi.data
for _ in range(nsteps):

    psi_exact = cite_step(
        psi_exact.data,
        Hm,
        constraints_mat,
        dt=dt
    )

    energies_exact.append(
        np.real(
            np.vdot(
                psi_exact,
                Hm @ psi_exact
            )
        )
    )

    for k,B in enumerate(C):

        constraints_history[k].append(
            np.real(
                np.vdot(
                    psi_exact,
                    B @ psi_exact
                )
            )
        )

KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energies = np.asarray(energies)
energies_exact = np.asarray(energies_exact)

iters = np.arange(len(energies))

fig, ax = plt.subplots(
    figsize=(8, 3.8),
    dpi=100
)

ax.plot(
    iters,
    energies_exact,
    lw=2.8,
    label="Exact CITE"
)

ax.plot(
    iters,
    energies,
    lw=2.5,
    ls="--",
    label="CQITE"
)

ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel(r"$\langle H\rangle$", fontsize=12)

ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.tick_params(direction="in")

plt.tight_layout()
fig.savefig("zz_exp.pdf", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
mag = np.asarray(magnetizations)

fig, ax = plt.subplots(
    figsize=(8,3.8)
)

for k in range(mag.shape[1]):
    ax.plot(
        np.abs(mag[:,k]-target),
        lw=2,
        label=rf"$C_{k+1}$"
    )

ax.set_yscale("log")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"$|\langle C_i\rangle-c|$")
ax.grid(alpha=.3)

ax.legend()

plt.tight_layout()

fig.savefig("zz_con.pdf", bbox_inches="tight", transparent=True)